# report02 — 드론 3D 모델 만들기

> ### ❓ 이 리포트가 답하는 질문
> **실험에 띄울 드론의 3D 모델을 어떻게 만들었고, 그 겉모양이 실제 기체와 맞는가?**

### ⚡ 결론부터 (TL;DR)

1. DJI 는 공식 3D 파일을 주지 않는다. 그래서 **공식 제원표의 숫자**(외곽 가로·세로·높이, 프로펠러 지름, 무게)만 가지고 코드가 형상을 **깎아 만든다** — 사진과 치수표로 프라모델을 깎듯이.
2. 5종의 드론을 만들었다(총 삼각형 **145,752개**). 만든 뒤 외곽 상자 치수를 공식값과 맞춰보면 오차가 **사실상 0**이다 — 이는 '설계 의도(공식 치수)가 결과물에 그대로 새겨졌다'는 뜻이다.
3. 전파로 보면 **플라스틱 껍데기보다 속의 금속이 훨씬 밝다.** 그래서 겉껍데기만이 아니라 **배터리·모터·기판** 같은 내부 금속까지 함께 넣는다 — 나중에 이 드론이 레이더에 얼마나 밝게 잡히는지를 정직하게 계산하려면 속이 있어야 한다.
4. 만든 메쉬는 전부 **품질 검사**를 통과한다 — 물 안 새는 닫힌 껍데기(watertight), 법선이 밖을 향함, 뒤집힌 면·찌부러진 면 0개(5종 전체에서 불량 면 **0개**).

### 🗺️ 어디부터 읽나

| 절 | 무엇을 |  |
|---|---|---|
| §1 | DJI 는 CAD 를 안 준다 → 제원표로 깎는다 | 문제 설정 · 비유 |
| §2 | 어떻게 깎나 — 단면 쌓기·쓸기·돌리기·붙이기 | 만드는 도구 |
| §3 | 겉모양이 맞나 — 외곽 상자 치수 대조 | 충실도 증거 |
| §4 | 5종 드론 갤러리 | 결과물 |
| §5 | 속이 더 중요하다 — 내부 금속을 넣는 이유 | RF 충실도 |
| §6 | 메쉬 품질 검사 — 새지 않는가 | 계산이 믿을 만한가 |

---


## 📋 이 결과가 어디서 어떻게 나왔나

> 이 절은 **직접 참여하지 않은 사람도 출처를 따라가고 재현할 수 있도록** 넣었습니다. 버전·GPU 는 노트북 생성 시점에 **실제로 읽어온 값**입니다.

### 1️⃣ 무엇을 참고했나

| 항목 | 출처 | 성격 |
|---|---|---|
| 외곽치수·대각거리·프로펠러·무게·회전수 | DJI 공식 제품 스펙 (기체별 제원표) | 공식 제원 |
| 드론 부위별 재질 · 반사계수 |Γ| | 문헌값(ABS/PC 유전율) · ITU-R P.2040(metal) · 내부 재질표 | 물성 기준 |
| 삼각형 수 · 부위 수 · 외곽 오차 · 검사 결과 | outputs/report1.json (meshes 블록) | 측정값(코드 산출) |

### 2️⃣ 어떤 도구가 무엇을 했나 — **Sionna 내부인가, 우리가 짠 건가**

| 도구 | 하는 일 | 어디서 도는가 |
|---|---|---|
| `trimesh-cad` | CAD 모델링 (`src/cadkit.py` + `src/drone_cad.py`) — 로프트·스윕·회전체·**불리언(CSG)** | 🔴 **별도** (trimesh + manifold3d + shapely + scipy, CPU) |
| `trimesh-check` | 메쉬 검증 (`src/mesh_check.py`) — watertight · winding · 법선방향 · 퇴화면 | 🔴 **별도** (trimesh, CPU). 빌드 게이트로 회귀를 막는다 |
| `matplotlib` | matplotlib — 도표·그래프 | 🔴 **별도** (CPU). 계산 결과를 *그리기만* 한다 |

> 🔑 **이 구분이 이 프로젝트에서 가장 자주 오해받는 지점입니다.**
> - **전파**(경로·지연·도플러·렌더·라디오맵)는 🟢 **Sionna 가** 합니다.
> - **표적 RCS** 는 🟡 우리가 짠 **SBR** 이 합니다 — Sionna 에 RCS 솔버가 없기 때문입니다. 다만 광선추적은 Sionna 가 쓰는 **Mitsuba 3 엔진을 그대로** 씁니다.
> - **레이더 신호처리**(ECA/CFAR)는 🔴 우리가 짰습니다 — Sionna 에 레이더 DSP 가 없습니다.

### 3️⃣ 라이브러리 (실행 시점 **실측** 버전)

| 라이브러리 | 버전 | 무엇에 쓰나 |
|---|---|---|
| `trimesh` | 4.12.2 | 메쉬 CAD·**검증** — 로프트/스윕/불리언 + watertight·법선·퇴화면 검사 |
| `manifold3d` | 3.5.2 | **불리언(CSG) 엔진** — trimesh 백엔드. 겹친 파트의 내부 면을 녹여 없앤다 |
| `shapely` | 2.1.2 | 2D 단면 폴리곤(버퍼·오프셋) → 로프트 입력 |
| `scipy` | 1.18.0 | 스플라인(단면 보간·암 경로) · STFT(스펙트로그램) |
| `numpy` | 2.5.0 | 수치 계산 전반 |
| `matplotlib` | 3.11.0 | 도표 |

### 4️⃣ 어디서 돌렸나

- **Python** 3.12.13 · Linux 5.15.0-136-generic
- **GPU** — `src/gpu.py` 가 **여유 메모리를 보고 자동 선택**합니다 (하드코딩 없음):
  - 0, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 1, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 2, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
  - 3, NVIDIA GeForce RTX 4090, 24564 MiB, 580.126.09
- `CUDA_VISIBLE_DEVICES` = (고정 안 함 — src/gpu.py 가 여유 메모리 보고 자동 선택)

- **계산 비용**: CPU 만 사용(메쉬 생성·검사는 GPU 불필요). 5종 전체 생성+검사 수십 초 규모.

### 5️⃣ 어떻게 다시 돌리나 (재현)

```bash
# 드론 메쉬를 만들고 검사·렌더까지 다시 돌린다
~/.venvs/py312/bin/python src/build_report1.py    # → outputs/report1.json (meshes 블록)
~/.venvs/py312/bin/python src/make_notebook02.py  # → report02.ipynb
```

### 6️⃣ 본문 숫자는 어디서 오나

이 노트북의 **숫자는 손으로 적지 않았습니다.** 메쉬 생성·검사 스크립트가 `outputs/report1.json` 의 `meshes` 블록에 삼각형 수·부위 수·외곽 오차·검사 결과를 남기고, 이 생성기가 그 값을 읽어 본문에 주입합니다. 숫자가 이상하면 그 JSON 을 보세요.

### 7️⃣ 무엇이 산출되나

| 산출물 | 무엇 |
|---|---|
| `outputs/figures/report1_cad_pipeline.png` | 형상을 깎는 순서 도식 |
| `outputs/figures/report1_envelope.png` | 외곽 상자 치수 대조 그림 |
| `outputs/figures/report1_meshcheck.png` | 메쉬 품질 검사 결과 그림 |
| `outputs/renders/r1_30_drone_*.png` | 5종 드론 3-뷰 렌더 |

### 8️⃣ ⚠️ 믿으면 안 되는 것 (신뢰 경계)

> 정직함이 이 프로젝트의 규칙입니다. **아래는 이 리포트가 보장하지 않는 것들입니다.**

- 이 모델은 **제원표의 치수와 눈에 보이는 형상**을 맞춘 것이다. 나사·배선·틈새 같은 밀리미터 이하 디테일은 없다 — 이 리포트는 '겉모양이 맞다'까지만 보장한다.
- **실물 사진과의 정성 대조**(정말 그 기체처럼 보이는가)는 여기서 하지 않는다 → report03 소관.
- **레이더 밝기(RCS, σ)** 는 이 리포트에서 계산하지 않는다. 여기서 만든 메쉬를 **재료**로 삼아 이후 리포트(06~08)가 광선+물리광학으로 밝기를 계산한다.

### 9️⃣ 앞뒤 리포트

| 리포트 | 관계 |
|---|---|
| report01 — 통제 환경: 반무향 챔버 | 이 드론들을 **띄울 무대**. 앞 리포트. |
| report03 — 모델을 믿어도 되나(실물 대조) | 여기서 만든 겉모양을 **실물과 대본다**. 다음 리포트. |
| report06~08 — RCS · SBR | 이 메쉬를 재료로 표적의 **레이더 밝기 σ** 를 계산한다. |

<details><summary><b>🔤 용어집 — 모르는 말이 나오면 여기</b> (클릭)</summary>

| 용어 | 뜻 |
|---|---|
| **메쉬(mesh)** | 3D 물체의 표면을 작은 **삼각형** 수만 개로 이어붙인 껍데기. 컴퓨터가 다루는 형상의 기본 형식. |
| **삼각형(면, face)** | 메쉬를 이루는 최소 조각. 많을수록 곡면이 매끈하지만 계산은 무거워진다. |
| **외곽 상자(bounding box)** | 물체를 딱 감싸는 최소 직육면체. 그 가로·세로·높이가 곧 제원표의 외곽치수. |
| **로프트(loft)** | 빵을 슬라이스로 쌓듯, 여러 **단면**을 위치별로 놓고 매끄럽게 이어 3D 덩어리를 만드는 것. |
| **스윕(sweep)** | 정해진 **경로**를 따라 단면을 밀고 나가며 만드는 것(예: 휘어진 팔·프로펠러 날). |
| **회전체(revolve)** | 옆모습 하나를 축 둘레로 **돌려** 만드는 것(예: 모터 벨, 렌즈). |
| **불리언 / CSG** | 두 덩어리를 **합치거나(union) 파내는(subtract)** 연산. 겹친 속면을 녹여 하나의 매끈한 표면으로. |
| **watertight(수밀)** | 구멍·틈 없이 **완전히 닫힌** 껍데기. 물을 부어도 안 새는 그릇. 전파 반사 계산의 전제. |
| **법선(normal)** | 각 삼각형이 **어느 쪽이 바깥인지** 가리키는 화살표. 뒤집히면 되쏘는 방향을 틀리게 계산한다. |
| **퇴화면(degenerate)** | 면적이 0에 가까운 **찌부러진 삼각형**. 계산을 망가뜨리므로 없어야 한다. |
| **|Γ| (반사계수)** | 그 표면이 전파를 되쏘는 정도. 0=투명, 1=완전 거울. 금속은 1에 가깝고 플라스틱은 낮다. |

</details>

---


## 🔰 5분이면 이해하는 이 리포트

레이더로 드론을 **잡으려면**, 먼저 시뮬레이터 안에 그 드론이 있어야 합니다. 문제는 하나입니다.
**DJI 는 자기 드론의 3D 설계 파일을 공개하지 않습니다.**

그래서 우리는 **프라모델을 깎는 사람**처럼 합니다. 프라모델 장인은 실물이 없어도, **사진 몇 장과 치수표**(가로 몇 mm, 프로펠러 몇 mm, 무게 몇 g)만 있으면 똑같이 깎아냅니다. 우리도 DJI 가 **공개하는 공식 제원표의 숫자**만 가지고, 코드로 형상을 한 조각씩 빚어냅니다.

빚는 방법은 네 가지 손동작으로 요약됩니다:

- **쌓기**(단면을 아래에서 위로 쌓아 동체를 만들고),
- **쓸기**(휘어진 경로를 따라 밀어 팔과 프로펠러 날을 만들고),
- **돌리기**(옆모습을 축 둘레로 돌려 모터와 렌즈를 만들고),
- **붙이기/파내기**(조각들을 하나의 매끈한 덩어리로 합칩니다).

다 깎은 뒤에는 **두 가지를 확인**합니다.

**첫째, 겉모양이 맞나?** 만든 모델을 딱 감싸는 상자의 치수를 공식 치수표와 비교합니다 — 치수표대로 깎았으니 오차는 사실상 0, 즉 **크기와 비율은 실물과 같습니다**.

**둘째, 속을 넣었나?** 레이더(전파)의 눈으로 보면 매끈한 플라스틱 껍데기는 사실 좀 **투명**합니다. 정작 밝게 되빛나는 건 **속에 든 금속** — 배터리 팩, 모터, 회로기판입니다. 야간에 자동차 헤드라이트에 반사판만 유독 번쩍이는 것과 같습니다. 그래서 우리는 껍데기 안에 이 금속 부품들도 함께 넣습니다. 그래야 나중에 '이 드론이 레이더에 얼마나 밝게 보이나'를 **정직하게** 계산할 수 있습니다.

마지막으로, 만든 껍데기가 **물 안 새는 그릇**인지(구멍·뒤집힌 면이 없는지) 자동으로 검사합니다. 새는 그릇이면 전파 되쏘기 계산이 엉키기 때문입니다.

> 한 줄 요약: **공식 치수표로 5종의 드론을 깎고, 겉치수를 맞추고, 속 금속을 넣고, 새지 않는지 검사했다.** 이 드론들이 앞으로 모든 실험의 표적입니다.

## §1. 왜 직접 깎아야 하나

게임 캐릭터나 영화 소품이라면 3D 파일을 사거나 내려받으면 됩니다. 그런데 실제 드론을 시뮬레이션하려면 **그 실제 드론의 형상**이 있어야 하고, DJI 같은 제조사는 경쟁·보안 문제로 **설계 3D 파일을 공개하지 않습니다.** 인터넷에 떠도는 무료 모델들은 대개 비율이 어긋나거나, 속이 텅 비어 있거나, 출처를 알 수 없습니다.

다행히 제조사가 **반드시 공개하는 것**이 있습니다 — **제원표(spec sheet)** 입니다. 여기엔 늘 이런 숫자가 있습니다:

- **외곽 치수**: 펼친 기체를 감싸는 상자의 가로 × 세로 × 높이 (mm)
- **대각거리**: 마주 보는 모터 축 사이 거리 (프레임 크기의 표준 지표)
- **프로펠러**: 지름과 날 수
- **무게**·**회전수**(호버링/최대) 등

이 숫자들이면 **프라모델을 깎기에 충분**합니다. 프라모델 장인이 실물 없이 사진과 치수만으로 축소 모형을 만들 듯, 우리는 코드로 이 숫자들에 맞춰 형상을 빚습니다. 이렇게 만들면 **비율과 크기가 출처(공식값)에 묶여 있어**, 어디서 왔는지 모를 인터넷 모델보다 훨씬 믿을 만합니다.

## §2. 어떻게 깎나 — 네 가지 손동작

형상을 빚는 데는 `trimesh`(메쉬 도구) · `shapely`(2D 단면) · `scipy`(부드러운 곡선 보간) · `manifold3d`(덩어리 합치기) 라는 파이썬 라이브러리를 씁니다. 겉보기엔 복잡하지만, 실제로 하는 일은 **네 가지 기본 동작**의 조합입니다. 점토를 빚는 손동작이라고 생각하면 됩니다.

| 손동작 | 무엇을 하나 | 일상 비유 | 드론의 어디 |
|---|---|---|---|
| **쌓기**(로프트) | 여러 **단면**을 높이별로 놓고 매끄럽게 잇는다 | 식빵 슬라이스를 쌓아 한 덩어리 | 동체·캐노피 |
| **쓸기**(스윕) | **경로**를 따라 단면을 밀고 나간다 | 치약을 짜며 관을 만든다 | 휘어진 팔·프로펠러 날 |
| **돌리기**(회전체) | 옆모습 하나를 축 둘레로 돌린다 | 물레로 도자기를 뽑는다 | 모터 벨·렌즈 |
| **붙이기/파내기**(불리언·CSG) | 두 덩어리를 합치거나 파낸다 | 찰흙 두 덩이를 눌러 하나로 | 부품 결합·구멍 |

특히 **붙이기(CSG)** 가 중요합니다. 조각들을 그냥 겹쳐 놓으면 껍데기 안에 **보이지 않는 속면**이 남아 전파 계산을 어지럽힙니다. CSG(불리언 합집합)는 겹친 부분의 속면을 **녹여 없애고**, 바깥 표면만 남은 하나의 매끈한 덩어리로 만들어 줍니다.

부위별로 이름표(그룹)를 붙여 조립합니다 — **동체(body) · 캐노피(canopy) · 팔(arm) · 모터(motor) · 프로펠러(prop) · 착륙장치(gear) · 카메라(camera) · 배터리(battery) · 기판(pcb) · 식별색(accent)**. 이 이름표는 뒤에서 부위마다 다른 **재질**(전파를 얼마나 되쏘는지)을 붙일 때 그대로 쓰입니다.

![CAD 파이프라인](outputs/figures/report1_cad_pipeline.png)

> 위 그림은 단면 하나가 어떻게 쌓기·다듬기를 거쳐 완성된 동체 표면이 되는지를 단계별로 보여줍니다.

## §3. 겉모양이 맞나 — 외곽 상자로 확인

형상을 다 깎았으면, 가장 먼저 **크기와 비율**이 실물과 같은지 봅니다. 방법은 단순합니다. 만든 모델을 **딱 감싸는 최소의 상자**(외곽 상자)를 재서, 그 가로·세로·높이를 **공식 제원표의 외곽치수와 비교**합니다.

옷을 맞출 때 치수표대로 재단했으면 다 만든 옷의 치수가 표와 같아야 하는 것과 같습니다. 우리도 치수표대로 깎았으니 오차가 **사실상 0**으로 나옵니다 — 이건 '용케 맞았다'가 아니라 **'설계 의도(공식 치수)가 결과물에 그대로 새겨졌다'는 검증**입니다.

| 드론 | 공식 외곽 (가로×세로×높이, mm) | 만든 모델 외곽 (mm) | 최대 오차 |
|---|---|---|---|
| DJI Mini 5 Pro | 255 × 181 × 91 | 255 × 181 × 91 | 1.6e-14% |
| DJI Mavic 4 Pro | 329 × 390 × 135 | 329 × 390 × 135 | 0% |
| DJI Matrice 4E | 307 × 388 × 150 | 307 × 388 × 150 | 0% |
| DJI Phantom 4 | 290 × 290 × 196 | 290 × 290 × 196 | 0% |
| DJI S1000+ | 1016 × 1016 × 380 | 1016 × 1016 × 380 | 0% |

여기에 더해, 우리가 **직접 입력하지 않은** 값 하나가 저절로 맞는지도 봅니다 — **모터 대각거리**입니다. 우리는 외곽치수만 맞췄을 뿐 대각거리는 넣은 적이 없는데, 완성된 모델에서 마주 보는 모터 사이를 재보면 공식 대각거리에 근접하게 나옵니다. 예컨대 Mavic 4 Pro 는 공식 대각 441 mm 에 대해 모델에서 439 mm, Matrice 4E 는 공식 439 mm 에 모델 431 mm 입니다. 입력하지 않은 치수가 저절로 맞는다는 건, 프레임의 **내부 비율까지** 실물과 일관된다는 뜻입니다.

![외곽 대조](outputs/figures/report1_envelope.png)

## §4. 만든 5종

이렇게 해서 크기와 용도가 서로 다른 **5종**을 만들었습니다. 손바닥만 한 250 g 미니 드론부터, 8개 로터에 9.5 kg 나 나가는 산업용 대형 헥사콥터까지 폭이 넓습니다. 표적이 작고 크고, 프로펠러가 많고 적음에 따라 레이더에 잡히는 양상이 달라지므로, 일부러 다양하게 갖췄습니다.

| 드론 | 로터 | 프로펠러 | 무게 | 삼각형 수 | 부위 수 |
|---|---|---|---|---|---|
| DJI Mini 5 Pro | 4 | 152 mm · 2날 | 250 g | 24,398 | 25 |
| DJI Mavic 4 Pro | 4 | 267 mm · 2날 | 1063 g | 25,676 | 21 |
| DJI Matrice 4E | 4 | 274 mm · 2날 | 1219 g | 28,714 | 26 |
| DJI Phantom 4 | 4 | 240 mm · 2날 | 1380 g | 27,806 | 25 |
| DJI S1000+ | 8 | 381 mm · 2날 | 9500 g | 39,158 | 55 |

5종을 합치면 삼각형이 총 **145,752개**입니다. 삼각형이 많을수록 곡면이 매끈해 전파 되쏘기를 정밀하게 계산할 수 있지만, 그만큼 계산이 무거워집니다 — 이 정도가 형상 충실도와 계산 부담의 균형점입니다.

아래는 각 드론을 세 방향(비스듬히 · 옆 · 위)에서 렌더한 모습입니다.


**DJI Mini 5 Pro**

| 비스듬히 | 옆 | 위 |
|---|---|---|
| ![mini5pro iso](outputs/renders/r1_30_drone_mini5pro_iso.png) | ![mini5pro side](outputs/renders/r1_30_drone_mini5pro_side.png) | ![mini5pro top](outputs/renders/r1_30_drone_mini5pro_top.png) |

![.](outputs/renders/anim/spin_mini5pro.gif)

<sub>Mini 5 Pro 3D 모델 회전.</sub>

**DJI Mavic 4 Pro**

| 비스듬히 | 옆 | 위 |
|---|---|---|
| ![mavic4pro iso](outputs/renders/r1_30_drone_mavic4pro_iso.png) | ![mavic4pro side](outputs/renders/r1_30_drone_mavic4pro_side.png) | ![mavic4pro top](outputs/renders/r1_30_drone_mavic4pro_top.png) |

![.](outputs/renders/anim/spin_mavic4pro.gif)

<sub>Mavic 4 Pro 3D 모델 회전(실측 실험용 드론).</sub>

**DJI Matrice 4E**

| 비스듬히 | 옆 | 위 |
|---|---|---|
| ![matrice4e iso](outputs/renders/r1_30_drone_matrice4e_iso.png) | ![matrice4e side](outputs/renders/r1_30_drone_matrice4e_side.png) | ![matrice4e top](outputs/renders/r1_30_drone_matrice4e_top.png) |

**DJI Phantom 4**

| 비스듬히 | 옆 | 위 |
|---|---|---|
| ![phantom4 iso](outputs/renders/r1_30_drone_phantom4_iso.png) | ![phantom4 side](outputs/renders/r1_30_drone_phantom4_side.png) | ![phantom4 top](outputs/renders/r1_30_drone_phantom4_top.png) |

![.](outputs/renders/anim/spin_phantom4.gif)

<sub>Phantom 4 3D 모델 회전.</sub>

**DJI S1000+**

| 비스듬히 | 옆 | 위 |
|---|---|---|
| ![s1000plus iso](outputs/renders/r1_30_drone_s1000plus_iso.png) | ![s1000plus side](outputs/renders/r1_30_drone_s1000plus_side.png) | ![s1000plus top](outputs/renders/r1_30_drone_s1000plus_top.png) |

![.](outputs/renders/anim/spin_s1000plus.gif)

<sub>S1000+ 3D 모델 회전(큰 헥사콥터).</sub>

### 색은 곧 재질 — 그리고 실제 크기 비교

위 렌더의 **색은 재질**입니다(모든 드론 공통 규약): **강청=금속**(모터·배터리 포일) · **회색=플라스틱**(셸·착륙장치) · **검정=탄소섬유**(암) · **주황=프로펠러** · **청록=카메라**(금속하우징+유리) · **초록=PCB**. 색만 보면 그 부위가 무슨 재질인지 알 수 있고, 이 재질이 그대로 RCS(되쏘는 밝기) 계산에 쓰입니다(→ report06·07).

**실제 크기 비교(같은 축척, 위에서 본 모습).** 5종은 크기가 크게 다릅니다 — Mini 5 Pro(275 mm)부터 S1000+(1045 mm)까지 대각 길이가 약 4배:

![drone size comparison](outputs/figures/drone_size_compare.png)

<sub>같은 축척으로 나란히 둔 5종 실루엣(재질색·스케일바 0.5 m). 크기가 다르면 되쏘는 밝기(RCS)도 달라진다 — 큰 S1000+ 가 작은 Mini 5 Pro 보다 훨씬 밝게 잡힌다.</sub>

**부위별로 따로 움직인다 — 분절(articulated) 메쉬.** 몸체와 프로펠러가 **독립적으로** 회전합니다. 아래는 5종이 몸체를 돌리며 동시에 프로펠러를 스핀시키는 모습입니다(같은 메쉬로 마이크로도플러 시뮬레이션을 할 수 있는 이유 → report08):

![five drones spinning](outputs/renders/anim/drone_gallery_row.gif)

<sub>5종 동시 회전 + 프로펠러 스핀(분절 메쉬). 몸체 자세와 블레이드 회전이 분리돼 있어, 실제 비행 중 프로펠러만 빠르게 도는 상황을 그대로 만들 수 있다.</sub>

## §5. 껍데기보다 속 — 왜 내부 금속을 넣나

여기서 흔히 놓치는 점이 있습니다. **눈에 보이는 것**과 **전파에 보이는 것**은 다릅니다.

우리 눈에 드론은 매끈한 플라스틱 몸통입니다. 그런데 레이더(전파)의 눈으로 보면 그 플라스틱 껍데기는 상당히 **반투명**합니다 — 전파의 일부가 껍데기를 통과합니다. 정작 강하게 되빛나는 건 **속에 든 금속**입니다. 밤길에 자동차 불빛을 받았을 때, 옷은 어둡고 **반사판만 번쩍**이는 것과 똑같은 이치입니다.

그래서 우리는 껍데기만 만들지 않고, 전파적으로 밝은 **내부 금속 부품**을 함께 넣습니다:

- **배터리 팩** — GHz 대역에서 파우치의 금속 포일은 사실상 금속판처럼 되쏩니다.
- **모터** — 구리 코일을 감은 금속 벨.
- **회로기판(PCB)** — FR-4 밑판에 넓은 **구리 접지면**이 깔려 있어 금속면처럼 반사합니다.

부위마다 얼마나 되쏘는지를 **반사계수 |Γ|**(0=투명, 1=완전 거울)로 나타내면, 겉과 속의 차이가 한눈에 보입니다:

| 부위 | 재질 | 반사계수 &#124;Γ&#124; |
|---|---|---|
| 동체·캐노피·착륙장치 | 플라스틱(ABS/PC) | 0.28 |
| 프로펠러 | 얇은 플라스틱 | 0.25 |
| 팔(arm) | 탄소섬유 | 0.90 |
| **모터 · 배터리** | **금속** | **0.85** |
| **기판(PCB)** | **구리 접지면** | **0.80** |

숫자로 보면 내부 금속(0.8~0.85)이 플라스틱 껍데기(0.28)보다 **세 배 가까이** 밝게 되쏩니다. 겉껍데기만 넣고 계산하면 표적이 실제보다 어둡게 나와, 나중에 '이 드론을 레이더로 잡을 수 있나'라는 질문에 **틀린 답**을 내게 됩니다. 그래서 속을 넣는 것이 선택이 아니라 필수입니다.

참고로 이 내부 금속 부품이 차지하는 삼각형은 DJI Mini 5 Pro 약 2,040개, DJI S1000+ 약 4,056개 수준으로, 겉모양뿐 아니라 속까지 형상을 갖췄음을 보여줍니다.

> 참고: 여기서 |Γ| 를 소개하는 이유는 **왜 속을 넣는지**를 설명하기 위해서입니다. 이 재질값으로 실제 표적 밝기(RCS, σ)를 계산하는 일은 report06~08 이 맡습니다.

## §6. 새지 않는 그릇인가 — 품질 검사

형상을 그럴듯하게 깎았다고 끝이 아닙니다. **전파 되쏘기 계산은 메쉬의 표면 상태에 민감**합니다. 특히 세 가지가 어긋나면 밝기 계산이 조용히 틀어집니다:

- **수밀(watertight)** — 껍데기에 **구멍이나 틈**이 있으면 안 됩니다. 물을 부어도 안 새는 닫힌 그릇이어야 합니다. 새는 곳이 있으면 그 자리에서 전파가 표면 안팎을 넘나들며 계산이 엉킵니다.
- **법선(normal) 방향** — 각 삼각형에는 '어느 쪽이 바깥'인지 가리키는 화살표가 있습니다. 이게 **안쪽을 향해 뒤집혀** 있으면, 되쏘는 방향을 반대로 계산해 표적이 어둡거나 밝게 잘못 나옵니다.
- **퇴화면(degenerate)** — 면적이 0에 가까운 **찌부러진 삼각형**은 계산에서 0으로 나누는 오류를 냅니다.

그래서 메쉬를 만들 때마다 `trimesh` 로 **자동 검사**를 겁니다. 부위별로 닫혀 있는지(watertight), 안쪽을 향한 법선이 몇 개인지, 와인딩(면의 방향 규칙)이 뒤집힌 면·찌부러진 면이 몇 개인지를 세어, **하나라도 있으면 빌드를 통과시키지 않습니다.** 자동차 공장에서 완성차마다 누수 검사를 하는 것과 같습니다.

결과는 깔끔합니다 — **5종 전체에서 안쪽 법선·역와인딩·퇴화면이 모두 0개**입니다. 모든 부위가 닫힌 껍데기이고, 모든 삼각형이 바깥을 봅니다. 즉 이후 리포트가 이 메쉬 위에서 하는 전파·밝기 계산이 형상 결함 때문에 틀어질 걱정은 없습니다.

![메쉬 검사](outputs/figures/report1_meshcheck.png)

---

> **앞 리포트**: report01 — 통제 환경(반무향 챔버). 이 드론들을 **띄울 무대**를 지었습니다.
> **다음 리포트**: report03 — 여기서 만든 겉모양을 **실물과 대조**해, 이 모델을 믿어도 되는지 확인합니다.